# Exercise 7: From the log to the lake

Every run of your job does three things: **read** the new records of the topic `logs`,
**write** them into the lake (Parquet files in `data/lake/`), **remember** where it stopped.

This notebook gives you the building blocks (`tools/lakelib.py`). You write one function,
`run_once()`. Save your final version of this notebook into your `solution/` folder.

In [1]:
import sys; sys.path.append("/home/jovyan/tools")
from lakelib import *

## 1. The example: a job that copies everything

It reads the topic from the beginning, every time. Run the next two cells and look at the
checker's last line.

In [2]:
def run_once():
    new = read_new(None)          # None = from the beginning of the topic
    write_files(new, LAKE)        # straight into the lake

reset()                           # empty lake
run_once()
check()

reset: lake, staging and state deleted
read_new: 6,564,000 new records
write_files: 6,564,000 records -> /home/jovyan/data/lake/logs/run-20261125-125003-9558d5

Lake check — topic 'logs' (4 partitions)   2026-11-25 12:51:55
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0         [0, 1,676,994)   1,676,994           0          0           0
        1         [0, 1,637,064)   1,637,064           0          0           0
        2         [0, 1,625,253)   1,625,253           0          0           0
        3         [0, 1,624,689)   1,624,689           0          0           0
-------------------------------------------------------------------------------
    total      6,564,000 records   6,564,000           0          0           0

files under /home/jovyan/data/lake: 4 data files, 342.4 MB, 85.61 MB pe

{'topic': 6564000,
 'lake': 6564000,
 'dup': 0,
 'miss': 0,
 'lag': 0,
 'files': 4,
 'mb': 342.442783,
 'ok': True,
 'problems': []}

In [3]:
run_once()                        # the next run
check()

read_new: 6,564,000 new records
write_files: 6,564,000 records -> /home/jovyan/data/lake/logs/run-20261125-125155-7c0047

Lake check — topic 'logs' (4 partitions)   2026-11-25 12:53:32
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0         [0, 1,676,994)   3,353,988   1,676,994          0           0
        1         [0, 1,637,064)   3,274,128   1,637,064          0           0
        2         [0, 1,625,253)   3,250,506   1,625,253          0           0
        3         [0, 1,624,689)   3,249,378   1,624,689          0           0
-------------------------------------------------------------------------------
    total      6,564,000 records  13,128,000   6,564,000          0           0

files under /home/jovyan/data/lake: 8 data files, 684.9 MB, 85.61 MB per file on average
check took 0.8 s

Exa

{'topic': 6564000,
 'lake': 13128000,
 'dup': 6564000,
 'miss': 0,
 'lag': 0,
 'files': 8,
 'mb': 684.885566,
 'ok': False,
 'problems': ['6,564,000 duplicates']}

## 2. The building blocks

| Block | What it does |
|---|---|
| `load_state()` | where the last run stopped, from a small state file (`None` if there is none) |
| `save_state(new.end)` | stores where this run stopped in the state file |
| `lake_position()` | where the last run stopped, read from the lake itself (`None` if the lake is empty) |
| `read_new(start)` | the records after `start`, up to the end of the topic *now* (`new.end`) |
| `write_files(new, folder)` | writes them as Parquet into a new run folder inside `LAKE` or `STAGING` |
| `publish(run)` | moves a finished run folder from `STAGING` into the lake: one rename, a single step |

`LAKE` is what readers see; `STAGING` is a folder next to it that nobody reads.
`new.df.show()` shows the records of a run as a Spark DataFrame.

## 3. Your architecture

Write `run_once()` for the architecture you chose (A, B or C, see the README).
The order of the calls *is* your architecture.

In [4]:
def run_once():
    # 1. where did the last run stop?
    start = lake_position()
    # 2. read what is new
    new = read_new(start)
    # 3. write it
    run = write_files(new, STAGING)
    # 4. remember where you stopped (if your architecture needs it)
    publish(run)

## 4. Check it

`check_all` empties the lake, then runs your job in five situations: a first run, new records,
nothing new, a run that stops **while** writing, a run that stops **after** writing.
It takes 2-4 minutes.

Architecture C: readers use a view, so check through it: `check_all(run_once, lake_sql=DEDUP_SQL)`.

In [5]:
check_all(run_once)

reset: lake, staging and state deleted

situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             6,564,000           0        0           0  97.4s  PASS
2 new records arrive, run again         6,664,000           0        0           0   6.8s  PASS
3 nothing new, run again                6,664,000           0        0           0   0.5s  PASS
4 run stops WHILE writing: right after  6,664,000           0        0     100,000         PASS
  ... then the next run                 6,764,000           0        0           0   6.7s  PASS
5 run stops AFTER writing: right after  6,764,000           0        0     100,000         PASS
  ... then the next run                 6,864,000           0        0           0   6.7s  PASS
-----------------------------------------------------------------------------------------------

[('1 first run',
  {'topic': 6564000,
   'lake': 6564000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 342.442783,
   'ok': True,
   'problems': []},
  97.42744851112366),
 ('2 new records arrive, run again',
  {'topic': 6664000,
   'lake': 6664000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 343.547128,
   'ok': True,
   'problems': []},
  6.795142889022827),
 ('3 nothing new, run again',
  {'topic': 6664000,
   'lake': 6664000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 343.547128,
   'ok': True,
   'problems': []},
  0.5072190761566162),
 ('4 run stops WHILE writing: right after',
  {'topic': 6764000,
   'lake': 6664000,
   'dup': 0,
   'miss': 0,
   'lag': 100000,
   'files': 8,
   'mb': 343.547128,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 6764000,
   'lake': 6764000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 12,
   'mb': 344.700701,
   'ok': True,
   'problems': []},
 